# 🚀 C-MAPSS FD001 – Streamlit Demo (Colab)

This notebook is **fully self-contained**. It will:
1. Clone the original repo for models.
2. Download the FD001 dataset.
3. Create `preprocess.py` and `app.py` on-the-fly.
4. Launch Streamlit + a **Cloudflare tunnel** (free, no signup).

**Just run all cells (`Runtime → Run all`).**

In [ ]:
# 1. Clone original repo for models
!git clone https://github.com/biswajitsahoo1111/rul_codes_open.git
%cd rul_codes_open

# 2. Download FD001 data
import os
os.makedirs('data', exist_ok=True)
!wget -q https://raw.githubusercontent.com/deniz-tuncbilek/Predictive_Maintenance_Demo/master/CMAPSSData/train_FD001.txt -O data/train_FD001.txt
!wget -q https://raw.githubusercontent.com/deniz-tuncbilek/Predictive_Maintenance_Demo/master/CMAPSSData/test_FD001.txt -O data/test_FD001.txt
!wget -q https://raw.githubusercontent.com/deniz-tuncbilek/Predictive_Maintenance_Demo/master/CMAPSSData/RUL_FD001.txt -O data/RUL_FD001.txt
print('✅ Repo + data ready')

In [ ]:
# 3. Install Streamlit (TensorFlow is pre-installed in Colab)
!pip install -q streamlit

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print("✅ Using Colab pre-installed TensorFlow")

In [ ]:
# 4. Create preprocess.py
preprocess_code = '''import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

WINDOW_LENGTH = 30
SHIFT = 1
EARLY_RUL = 125
NUM_TEST_WINDOWS = 5
COLUMNS_TO_DROP = [0, 1, 2, 3, 4, 5, 9, 10, 14, 20, 22, 23]

def process_targets(data_length, early_rul=None):
    if early_rul is None:
        return np.arange(data_length - 1, -1, -1)
    else:
        early_rul_duration = data_length - early_rul
        if early_rul_duration <= 0:
            return np.arange(data_length - 1, -1, -1)
        else:
            return np.append(early_rul * np.ones(shape=(early_rul_duration,)), np.arange(early_rul - 1, -1, -1))

def process_input_data_with_targets(input_data, target_data=None, window_length=1, shift=1):
    num_batches = int(np.floor((len(input_data) - window_length) / shift)) + 1
    num_features = input_data.shape[1]
    output_data = np.repeat(np.nan, repeats=num_batches * window_length * num_features).reshape(num_batches, window_length, num_features)
    if target_data is None:
        for batch in range(num_batches):
            output_data[batch, :, :] = input_data[(0 + shift * batch):(0 + shift * batch + window_length), :]
        return output_data
    else:
        output_targets = np.repeat(np.nan, repeats=num_batches)
        for batch in range(num_batches):
            output_data[batch, :, :] = input_data[(0 + shift * batch):(0 + shift * batch + window_length), :]
            output_targets[batch] = target_data[(shift * batch + (window_length - 1))]
        return output_data, output_targets

def process_test_data(test_data_for_an_engine, window_length, shift, num_test_windows=1):
    max_num_test_batches = int(np.floor((len(test_data_for_an_engine) - window_length) / shift)) + 1
    if max_num_test_batches < num_test_windows:
        required_len = (max_num_test_batches - 1) * shift + window_length
        batched = process_input_data_with_targets(test_data_for_an_engine[-required_len:, :], target_data=None, window_length=window_length, shift=shift)
        return batched, max_num_test_batches
    else:
        required_len = (num_test_windows - 1) * shift + window_length
        batched = process_input_data_with_targets(test_data_for_an_engine[-required_len:, :], target_data=None, window_length=window_length, shift=shift)
        return batched, num_test_windows

def load_and_preprocess(train_path, test_path, rul_path):
    train_data = pd.read_csv(train_path, sep=r"\s+", header=None)
    test_data = pd.read_csv(test_path, sep=r"\s+", header=None)
    true_rul = pd.read_csv(rul_path, sep=r"\s+", header=None)
    train_data_first_column = train_data[0]
    test_data_first_column = test_data[0]
    scaler = StandardScaler()
    train_data_scaled = scaler.fit_transform(train_data.drop(columns=COLUMNS_TO_DROP))
    test_data_scaled = scaler.transform(test_data.drop(columns=COLUMNS_TO_DROP))
    train_data = pd.DataFrame(data=np.c_[train_data_first_column, train_data_scaled])
    test_data = pd.DataFrame(data=np.c_[test_data_first_column, test_data_scaled])
    num_test_machines = len(test_data[0].unique())
    processed_test_data = []
    num_test_windows_list = []
    for i in np.arange(1, num_test_machines + 1):
        temp_test_data = test_data[test_data[0] == i].drop(columns=[0]).values
        if len(temp_test_data) < WINDOW_LENGTH:
            raise AssertionError(f"Test engine {i} doesn't have enough data for window_length of {WINDOW_LENGTH}")
        test_data_for_an_engine, num_windows = process_test_data(temp_test_data, window_length=WINDOW_LENGTH, shift=SHIFT, num_test_windows=NUM_TEST_WINDOWS)
        processed_test_data.append(test_data_for_an_engine)
        num_test_windows_list.append(num_windows)
    processed_test_data = np.concatenate(processed_test_data)
    true_rul = true_rul[0].values
    return processed_test_data, true_rul, scaler, num_test_windows_list
'''
with open('preprocess.py', 'w') as f:
    f.write(preprocess_code)
print('✅ preprocess.py created')

In [ ]:
# 5. Create app.py
app_code = '''import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

st.set_page_config(page_title="C-MAPSS FD001 – Turbofan RUL Predictor", page_icon="🔧", layout="wide")
st.title("🔧 NASA C-MAPSS FD001 – Turbofan Engine RUL Prediction")
st.markdown("""
This interactive demo showcases a **pre-trained LSTM** model for predicting
**Remaining Useful Life (RUL)** of turbofan engines using the NASA C-MAPSS dataset.

**Model:** LSTM with piecewise-linear degradation (RMSE ≈ 15.17 cycles)  
**Dataset:** FD001 (100 test engines, single operating condition, HPC degradation)
""")

st.sidebar.header("📊 Model Details")
st.sidebar.markdown("""
| Property | Value |
|---|---|
| Architecture | LSTM (128→64→32) + Dense(96→128→1) |
| Window length | 30 cycles |
| Features | 14 sensors/settings |
| Degradation | Piecewise linear (early RUL = 125) |
| Test windows / engine | 5 (averaged) |
| **RMSE** | **15.17 cycles** |
| Loss function | MSE |
| Optimizer | Adam (lr = 0.001) |
""")
st.sidebar.info("The LSTM outperforms the XGBoost piecewise baseline (RMSE ≈ 19.78).")

@st.cache_resource(show_spinner="Loading LSTM model …")
def load_model():
    import tensorflow as tf
    from tensorflow.keras import layers
    import os
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    
    # Rebuild the exact architecture from the notebook
    # (avoids Keras 2 vs 3 config incompatibility with time_major)
    model = tf.keras.Sequential([
        layers.LSTM(128, input_shape=(30, 14), return_sequences=True, activation="tanh"),
        layers.LSTM(64, activation="tanh", return_sequences=True),
        layers.LSTM(32, activation="tanh"),
        layers.Dense(96, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1)
    ])
    
    model_path = Path(__file__).parent / "saved_models" / "cmapss" / "FD001_LSTM_piecewise_RMSE_15.1655.h5"
    model.load_weights(str(model_path))
    return model

@st.cache_data(show_spinner="Preprocessing C-MAPSS data …")
def load_data():
    from preprocess import load_and_preprocess
    data_dir = Path(__file__).parent / "data"
    return load_and_preprocess(data_dir / "train_FD001.txt", data_dir / "test_FD001.txt", data_dir / "RUL_FD001.txt")

try:
    model = load_model()
    processed_test_data, true_rul, scaler, num_test_windows_list = load_data()
except Exception as e:
    st.error(f"Error loading model or data: {e}")
    st.stop()

predictions_all = model.predict(processed_test_data, verbose=0).flatten()
predicted_rul = []
idx = 0
for n_win in num_test_windows_list:
    predicted_rul.append(predictions_all[idx : idx + n_win].mean())
    idx += n_win
predicted_rul = np.array(predicted_rul)

rmse = np.sqrt(np.mean((predicted_rul - true_rul) ** 2))
mae = np.mean(np.abs(predicted_rul - true_rul))

st.subheader("📈 Overall Test-Set Performance")
col1, col2, col3 = st.columns(3)
col1.metric("RMSE", f"{rmse:.2f} cycles", delta="vs 15.17 reported")
col2.metric("MAE", f"{mae:.2f} cycles")
col3.metric("Test engines", len(true_rul))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(true_rul, predicted_rul, alpha=0.6, edgecolors="k", s=60)
max_rul = max(true_rul.max(), predicted_rul.max())
ax.plot([0, max_rul], [0, max_rul], "r--", lw=2, label="Perfect prediction")
ax.set_xlabel("True RUL (cycles)")
ax.set_ylabel("Predicted RUL (cycles)")
ax.set_title("True vs Predicted RUL – FD001 Test Set")
ax.legend()
ax.grid(True, alpha=0.3)
st.pyplot(fig)

residuals = predicted_rul - true_rul
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.hist(residuals, bins=20, edgecolor="k", alpha=0.7)
ax2.axvline(0, color="r", linestyle="--", lw=2)
ax2.set_xlabel("Residual (Predicted – True RUL)")
ax2.set_ylabel("Count")
ax2.set_title("Residual Distribution")
ax2.grid(True, alpha=0.3)
st.pyplot(fig2)

st.subheader("🔍 Per-Engine RUL Explorer")
engine_id = st.selectbox("Select test engine", np.arange(1, len(true_rul) + 1))
eng_idx = engine_id - 1
st.markdown(f"""
| Metric | Value |
|---|---|
| **True RUL** | {true_rul[eng_idx]:.1f} cycles |
| **Predicted RUL** | {predicted_rul[eng_idx]:.1f} cycles |
| **Error** | {predicted_rul[eng_idx] - true_rul[eng_idx]:.1f} cycles |
""")

st.subheader("📋 Performance Summary Table")
results_df = pd.DataFrame({
    "Engine": np.arange(1, len(true_rul) + 1),
    "True RUL": true_rul,
    "Predicted RUL": np.round(predicted_rul, 2),
    "Error": np.round(predicted_rul - true_rul, 2),
})
tab1, tab2 = st.tabs(["Worst errors (absolute)", "All engines"])
with tab1:
    worst = results_df.reindex(results_df["Error"].abs().sort_values(ascending=False).index).head(10)
    st.dataframe(worst.reset_index(drop=True), use_container_width=True)
with tab2:
    st.dataframe(results_df, use_container_width=True)

st.caption("Built on the open-source notebooks from biswajitsahoo1111/rul_codes_open using NASA C-MAPSS FD001.")
'''
with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py created')

In [ ]:
# 6. Kill any old Streamlit and start fresh
import os, subprocess, time
os.system('killall -9 streamlit 2>/dev/null')
time.sleep(1)

proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501',
     '--server.headless', 'true', '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(12)

# Quick health check
import urllib.request
try:
    urllib.request.urlopen('http://localhost:8501')
    print('✅ Streamlit is running on port 8501')
except Exception as e:
    print('❌ Streamlit did not start:', e)
    out, err = proc.communicate(timeout=2)
    print('STDERR:', err.decode()[:1000])

In [ ]:
# 7. Download Cloudflared and start a free tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, time, re

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if line:
        print(line.strip())
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(1)

if public_url:
    print('\n🌐 PUBLIC URL (click to open):')
    print(public_url)
else:
    print('Could not extract tunnel URL. Check logs above.')